# HSE RecSys Challenge 2026

Нужно рекомендовать каждому пользователю 10 фильмов. Метрика — **recall@10**.

Решение сочетает общий вкус пользователя и его недавнюю историю: **EASE + модель следующего фильма**.
Ниже — данные, гипотезы, проверка на трёх временных срезах и обучение на всех событиях.
Run All пересчитывает основные метрики и сохраняет submission.csv. CSV с данными лежат рядом с ноутбуком.

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['OMP_NUM_THREADS'] = '4'

import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import scipy.linalg as la
from IPython.display import display

DATA = Path('.')
K = 10
PARAMS = dict(global_lam=700, local_lam=100, window=10,
              ridge_lam=100, tau=3, horizon=12, beta=64)
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.precision', 5)

events = pd.read_csv(DATA / 'events.csv')
users = pd.read_csv(DATA / 'user_features.csv').sort_values('user_id').reset_index(drop=True)
items = pd.read_csv(DATA / 'item_features.csv').sort_values('item_id').reset_index(drop=True)
sample = pd.read_csv(DATA / 'submission_sample.csv')
N_USERS, N_ITEMS = len(users), len(items)
print(f'numpy={np.__version__}, pandas={pd.__version__}, scipy={scipy.__version__}')


numpy=2.5.4, pandas=3.1.0, scipy=1.18.1


## 1. Что есть в данных

У каждого пользователя есть история. Повторных пар «пользователь — фильм» нет, поэтому уже оценённые
фильмы исключаем из рекомендаций. Матрицу взаимодействий делаем бинарной: низкая оценка тоже означает,
что пользователь выбрал этот фильм.

По пояснению организатора timestamp сохраняет порядок событий. Это не календарное время.
Пропуски в нумерации сами по себе не показывают, какие фильмы попадут в тест.

In [2]:
assert not events.isna().any().any()
assert not events.duplicated(['user_id', 'item_id']).any()
assert not events.duplicated(['user_id', 'timestamp']).any()
assert np.array_equal(users.user_id.values, np.arange(N_USERS))
assert np.array_equal(items.item_id.values, np.arange(N_ITEMS))
assert set(events.user_id) == set(users.user_id) == set(sample.user_id)
assert set(events.item_id) <= set(items.item_id)

history = events.groupby('user_id').timestamp.agg(['min', 'max', 'size'])
display(pd.Series({
    'Оценок': len(events), 'Пользователей': N_USERS, 'Фильмов в признаках': N_ITEMS,
    'Фильмов с оценками': events.item_id.nunique(),
    'Минимальная история': history['size'].min(),
    'Медианная история': history['size'].median(),
    'Максимальная история': history['size'].max(),
    'Пропусков внутри нумерации': (history['max'] + 1 - history['size']).sum(),
}, name='Значение').to_frame())
display(events.rating.value_counts(normalize=True).sort_index().rename('Доля оценок').to_frame())


,Значение
Оценок,894149.0
Пользователей,6040.0
Фильмов в признаках,3706.0
Фильмов с оценками,3690.0
Минимальная история,14.0
Медианная история,84.5
Максимальная история,2056.0
Пропусков внутри нумерации,98628.0


,Доля оценок
rating,
1,0.05621
2,0.10737
3,0.26150
4,0.34913
5,0.22580


## 2. Гипотезы и валидация

| Гипотеза | Проверка |
|---|---|
| Общие предпочтения важнее одной популярности | Популярное против EASE |
| Последние просмотры описывают ближайший интерес | Соседи пропусков против недавней истории |
| Направление последовательности полезно | Локальная EASE против предсказания следующего фильма |
| Общий вкус дополняет недавний контекст | Последовательная модель отдельно и в смеси с EASE |
| Способ смешивания влияет на результат | Сумма скоров против объединения рангов |

Точная схема скрытого теста не опубликована. Для выбора модели используем временную проверку:
скрываем последний известный фильм каждого пользователя и предсказываем его по прошлому.
Повторяем проверку для предпоследнего и третьего с конца фильма; все события после цели тоже удаляем.

В каждом срезе по одной цели на пользователя, поэтому recall@10 равен доле попаданий в десятку.
Timestamp не перенумеровываем. Все матрицы обучаем заново только на разрешённой истории.
Срезы пересекаются по пользователям: это проверка устойчивости, а не три независимых теста.

In [3]:
def to_matrix(df, values=None):
    values = np.ones(len(df), np.float32) if values is None else np.asarray(values, np.float32)
    return sp.csr_matrix((values, (df.user_id, df.item_id)), shape=(N_USERS, N_ITEMS))


def temporal_split(df, cut=1):
    d = df.sort_values(['user_id', 'timestamp'])
    from_end = d.groupby('user_id').cumcount(ascending=False)
    train = d[from_end >= cut].reset_index(drop=True)
    target = d[from_end == cut - 1].reset_index(drop=True)
    assert len(target) == N_USERS
    assert (train.groupby('user_id').timestamp.max()
            < target.set_index('user_id').timestamp).all()
    return train, target


def topk(scores, X, k=K):
    scores = np.array(scores, dtype=np.float32, copy=True)
    scores[X.nonzero()] = -np.inf
    top = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.take_along_axis(scores, top, 1).argsort(1)[:, ::-1]
    return np.take_along_axis(top, order, 1)


def recall_per_user(rec, target):
    truth = to_matrix(target).toarray().astype(bool)
    hits = truth[np.arange(N_USERS)[:, None], rec].sum(1)
    return hits / truth.sum(1)


## 3. Общие предпочтения: EASE

EASE учится по совместным оценкам фильмов. Для бинарной матрицы $X$:
$P=(X^\top X+\lambda I)^{-1}$, $B_{ij}=-P_{ij}/P_{jj}$, $B_{ii}=0$.
Скор пользователя — $XB$. Используем $\lambda=700$.
Это персонализированный ориентир для сравнения с популярностью.

In [4]:
def ease_B(X, lam=700):
    gram = (X.T @ X).toarray().astype(np.float64)
    gram[np.diag_indices(X.shape[1])] += lam
    precision = np.linalg.inv(gram)
    B = -precision / np.diag(precision)
    np.fill_diagonal(B, 0)
    return B.astype(np.float32)


## 4. Ближайший контекст: локальная EASE

Обучаем EASE на окнах из 10 позиций. Усредняем статистику окон со сдвигами 0 и 5,
чтобы близкие фильмы не разделялись произвольной границей окна.

Сравниваем два контекста: соседи пропусков в нумерации и последние просмотры.
Во втором случае вес события — $\exp(-d/\tau)$, где $d$ — расстояние от последней позиции,
$\tau=3$. Это расстояние в порядке оценок, а не в днях.

In [5]:
def window_matrix(df, length=10, offset=0):
    d = df.sort_values(['user_id', 'timestamp'])
    key = pd.factorize(d.user_id.values.astype(np.int64) * 100_000
                       + (d.timestamp.values + offset) // length)[0]
    return sp.csr_matrix((np.ones(len(d), np.float32), (key, d.item_id.values)),
                         shape=(key.max() + 1, N_ITEMS))


def local_ease_B(df, length=10, lam=100):
    windows = [window_matrix(df, length, offset) for offset in (0, length // 2)]
    gram = sum((w.T @ w).toarray().astype(np.float64) for w in windows) / 2
    gram[np.diag_indices(N_ITEMS)] += lam
    precision = np.linalg.inv(gram)
    B = -precision / np.diag(precision)
    np.fill_diagonal(B, 0)
    return B.astype(np.float32)


def tail_weights(df, tau=3, horizon=None):
    distance = (df.groupby('user_id').timestamp.transform('max') - df.timestamp).to_numpy()
    weights = np.exp(-distance / tau)
    if horizon is not None:
        weights *= distance < horizon
    return to_matrix(df, weights)


def gap_weights(df, tau=2, tail_weight=0.5):
    rows, cols, values = [], [], []
    for u, g in df.sort_values('timestamp').groupby('user_id'):
        pos = g.timestamp.values
        present = np.zeros(pos.max() + 2, bool)
        present[pos] = True
        gaps = np.flatnonzero(~present)
        weights = np.exp(-(np.abs(pos[:, None] - gaps[None, :]) - 1) / tau)
        weights[:, -1] *= tail_weight
        rows.append(np.full(len(pos), u))
        cols.append(g.item_id.values)
        values.append(weights.sum(1))
    return sp.csr_matrix((np.concatenate(values).astype(np.float32),
                          (np.concatenate(rows), np.concatenate(cols))),
                         shape=(N_USERS, N_ITEMS))


## 5. Предсказание следующего фильма

Локальная EASE знает, какие фильмы встречаются рядом, но не различает прошлое и будущее.
Проверим направленную модель.

Для каждой известной оценки берём только предыдущие события того же пользователя в пределах 12 позиций.
Чем ближе событие к цели, тем больше его вес: $\exp(-(\Delta t-1)/3)$.
Матрица $C$ содержит такие контексты, а $Y$ — единицу на месте целевого фильма.

Решаем ridge-регрессию:
$$
B=(C^\top C+100I)^{-1}C^\top Y.
$$
После решения обнуляем диагональ. При рекомендации используем последние 12 позиций истории.
Уже оценённые фильмы исключаем перед выбором топ-10.

In [6]:
def directed_ridge(df, tau=3, lam=100, horizon=12):
    d = df.sort_values(['user_id', 'timestamp'])
    user, item, pos = d.user_id.to_numpy(), d.item_id.to_numpy(), d.timestamp.to_numpy()
    n = len(d)
    rows, cols, values = [], [], []
    for offset in range(1, horizon + 1):
        previous = np.arange(n - offset)
        target = previous + offset
        same_context = ((user[previous] == user[target])
                        & ((pos[target] - pos[previous]) <= horizon))
        previous, target = previous[same_context], target[same_context]
        rows.append(target)
        cols.append(item[previous])
        values.append(np.exp(-((pos[target] - pos[previous]) - 1) / tau).astype(np.float32))
    C = sp.csr_matrix((np.concatenate(values), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(n, N_ITEMS))
    Y = sp.csr_matrix((np.ones(n, np.float32), (np.arange(n), item)), shape=(n, N_ITEMS))
    gram = (C.T @ C).toarray().astype(np.float64)
    cross = (C.T @ Y).toarray().astype(np.float64)
    gram[np.diag_indices(N_ITEMS)] += lam
    B = la.cho_solve(la.cho_factor(gram, lower=True, check_finite=False),
                    cross, check_finite=False).astype(np.float32)
    np.fill_diagonal(B, 0)
    return B


## 6. Сравнение моделей

Финальный скор: **EASE + 64 × directed ridge**. Общая история дополняет краткий контекст.
Для сравнения берём локальную EASE, смесь по соседям пропусков и RRF — сумму обратных рангов
двух последовательных моделей. В RRF используем топ-200 и слагаемое $1/(20+\text{rank})$.

Параметры зафиксированы в PARAMS. Подробные результаты проверок сохранены в experiments.csv.
Ниже заново обучаем основные модели на каждом срезе.

In [7]:
def rrf(first, second, X):
    scores = np.zeros_like(first)
    for source in (first, second):
        rec = topk(source, X, k=200)
        np.add.at(scores, (np.arange(N_USERS)[:, None], rec),
                  1 / (20 + np.arange(1, 201)[None, :]))
    return scores


MODEL_NAMES = {
    'popularity': 'Популярное',
    'global_EASE': 'EASE',
    'current_gap_blend': 'EASE + соседи пропусков',
    'tail_local_tau3_beta16': 'EASE + недавняя локальная EASE',
    'directed_ridge': 'Directed ridge',
    'rrf_local_directed': 'RRF двух последовательных моделей',
    'global_plus_directed': 'EASE + directed ridge',
}


In [8]:
results, per_user = [], {}
started = time.time()
for cut in (1, 2, 3):
    train, target = temporal_split(events, cut)
    X = to_matrix(train)
    global_score = np.asarray(X @ ease_B(X, PARAMS['global_lam']))
    B_local = local_ease_B(train, PARAMS['window'], PARAMS['local_lam'])
    local_score = np.asarray(tail_weights(train, PARAMS['tau']) @ B_local)
    B_next = directed_ridge(train, PARAMS['tau'], PARAMS['ridge_lam'], PARAMS['horizon'])
    next_score = np.asarray(tail_weights(train, PARAMS['tau'], PARAMS['horizon']) @ B_next)

    def evaluate(name, scores):
        values = recall_per_user(topk(scores, X), target)
        per_user[(cut, name)] = values
        results.append({'cut': cut, 'model': name, 'recall10': values.mean()})

    pop = np.asarray(X.sum(0)).ravel()
    evaluate('popularity', np.tile(pop, (N_USERS, 1)))
    evaluate('global_EASE', global_score)
    evaluate('current_gap_blend', global_score + 2 * np.asarray(gap_weights(train) @ B_local))
    evaluate('tail_local_tau3_beta16', global_score + 16 * local_score)
    evaluate('directed_ridge', next_score)
    evaluate('rrf_local_directed', rrf(local_score, next_score, X))
    evaluate('global_plus_directed', global_score + PARAMS['beta'] * next_score)
    print(f'Срез {cut}: готово, прошло {time.time() - started:.0f} с')
    del X, B_local, B_next, global_score, local_score, next_score

validation = pd.DataFrame(results)
validation.to_csv('validation_results.csv', index=False)
comparison = validation.pivot(index='model', columns='cut', values='recall10')
comparison['mean'] = comparison.mean(axis=1)
display(comparison.reindex(MODEL_NAMES).rename(index=MODEL_NAMES))


Срез 1: готово, прошло 26 с


Срез 2: готово, прошло 53 с


Срез 3: готово, прошло 79 с


cut,1,2,3,mean
model,,,,
Популярное,0.03940,0.03626,0.03974,0.03847
EASE,0.08742,0.08477,0.09404,0.08874
EASE + соседи пропусков,0.08808,0.08841,0.09387,0.09012
EASE + недавняя локальная EASE,0.24470,0.24834,0.25182,0.24829
Directed ridge,0.24851,0.25497,0.26175,0.25508
RRF двух последовательных моделей,0.25381,0.26209,0.26507,0.26032
EASE + directed ridge,0.25298,0.26573,0.26821,0.26231


Недавняя история даёт основной прирост: смесь с локальной EASE получает в среднем **0.2483**
против **0.0901** у смеси по соседям пропусков.
Directed ridge отдельно достигает **0.2551**, а вместе с EASE — **0.2623**.

RRF даёт близкий результат — **0.2603**. Разница небольшая, поэтому не считаем преимущество
одного способа смешивания доказанным. Для финала выбираем сумму скоров с лучшим средним результатом.
Эта оценка относится к нашей валидации; значение на скрытом тесте считается отдельно.

## 7. Дополнительные проверки

Чтобы не перегружать ноутбук, подробные прогоны вынесены в experiments.csv.
В журнале указан протокол: temporal_oneout — временные срезы, random_10pct — случайное скрытие 10% истории.
Числа разных протоколов напрямую не сравниваем.

- **Бейзлайны и признаки.** На случайном сплите популярное дало 0.0692, популярное по демографии — 0.0741,
  по жанровому профилю — 0.0679. ItemKNN и iALS достигли 0.1495 и 0.1666, EASE — 0.1851.
- **Оценки как веса.** Бинарная EASE дала 0.1851, веса 1–5 — 0.1657. Удвоение высоких оценок
  и центрирование относительно среднего пользователя тоже не улучшили бинарный вариант.
- **Контекст и нормировки.** Проверены длина и сдвиги окон, затухание, вес предполагаемого хвоста,
  нормировка контекста, поправка на популярность, уменьшение веса активных пользователей и
  удаление отрицательных коэффициентов. Выигрыш отдельных настроек зависит от протокола.
- **Ранкер.** Проверены LightGBM с 17 признаками, последовательное добавление признаков
  и несколько разбиений. Его метрики в журнале относятся к случайному скрытию.
  Добавление признаков по одному не отделяет их собственный вклад от взаимодействий.
- **Последовательность.** Проверены двустороннее восстановление контекста, направленная ridge-модель,
  веса недавних событий и RRF. Финал выбираем по временной проверке из раздела 6.

Журнал содержит сохранённые результаты этих дополнительных прогонов.
Run All пересчитывает основные сравнения и финальный сабмит; весь поиск параметров повторно не запускается.

In [9]:
experiments = pd.read_csv('experiments.csv')
assert set(experiments.protocol) <= {'random_10pct', 'temporal_oneout'}
print(f'В подробном журнале {len(experiments)} измерений.')
display(experiments.groupby(['protocol', 'family'], sort=False).size().rename('Прогонов').to_frame())


В подробном журнале 203 измерений.


Прогонов
protocol        family                               
random_10pct    linear_models                      57
                ranker                             12
temporal_oneout context_grid                       55
                next_item                          23
random_10pct    model_grid                         10
                baselines                           3
                rating_weights                      4
                ranker_incremental_features        17
                ranker_inner_split                  3
                local_window_grid                  18
                gap_global                          1

## 8. Финальное обучение и сабмит

Обучаем EASE и directed ridge на всех доступных событиях. Считаем итоговые скоры, убираем
оценённые фильмы и выбираем 10 рекомендаций. Порядок пользователей берём из образца сабмита.

Результат — submission.csv с колонками user_id и item_id. Во второй колонке ID фильмов
разделены пробелами. Метрика Kaggle для этого файла пока не внесена.

In [10]:
started = time.time()
X = to_matrix(events)
global_score = np.asarray(X @ ease_B(X, PARAMS['global_lam']))
B_next = directed_ridge(events, PARAMS['tau'], PARAMS['ridge_lam'], PARAMS['horizon'])
next_score = np.asarray(tail_weights(events, PARAMS['tau'], PARAMS['horizon']) @ B_next)
recommendations = topk(global_score + PARAMS['beta'] * next_score, X)

submission = pd.DataFrame({
    'user_id': np.arange(N_USERS),
    'item_id': [' '.join(map(str, row)) for row in recommendations],
}).set_index('user_id').loc[sample.user_id].reset_index()

seen = set(zip(events.user_id, events.item_id))
assert submission.columns.tolist() == sample.columns.tolist()
assert submission.user_id.tolist() == sample.user_id.tolist()
assert submission.user_id.is_unique and len(submission) == N_USERS
assert not submission.isna().any().any()
assert recommendations.shape == (N_USERS, K)
assert all(len(set(row)) == K for row in recommendations)
assert recommendations.min() >= 0 and recommendations.max() < N_ITEMS
assert not any((u, int(i)) in seen for u, row in enumerate(recommendations) for i in row)

submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(submission)} пользователей × {K} фильмов; {time.time() - started:.0f} с')
display(submission.head())


submission.csv: 6040 пользователей × 10 фильмов; 13 с


,user_id,item_id
0,0,3002 1332 213 1811 1527 1461 2732 331 1044 2256
1,1,232 1246 3656 1039 2468 1868 1686 1491 3046 3101
2,2,2354 3358 1956 452 2342 3012 438 463 488 3238
3,3,183 3022 3656 304 3390 3441 169 2169 2342 3002
4,4,1545 802 1337 2338 2653 488 1560 1809 702 2484


## Итог

Ключевой сигнал — последние события и их порядок. Общий вкус пользователя полезен как дополнение.
Финальная модель остаётся простой: две линейные матрицы, фиксированные параметры и проверка
на трёх временных срезах. Для воспроизведения достаточно установить зависимости и выполнить ноутбук целиком.